In [1]:
import pandas as pd
import numpy as np
import joblib
import json
import os
from sklearn.preprocessing import LabelEncoder

# Load the RAW data again (same source as Phase 4)
df = pd.read_csv("data/raw/ecommerce_purchase_intent.csv")
df = df.drop_duplicates()

X = df.drop("Converted", axis=1)
y = df["Converted"]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("Categorical columns:", categorical_cols)

# Fit ONE encoder per column, save each one in a dictionary
encoders = {}
X_encoded = X.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    encoders[col] = le   # <-- keep a reference to this column's own encoder

print("Encoders saved for:", list(encoders.keys()))

Categorical columns: ['Month', 'VisitorType']
Encoders saved for: ['Month', 'VisitorType']


In [2]:
existing_train = pd.read_csv("data/processed/train.csv")
existing_test = pd.read_csv("data/processed/test.csv")

# Rebuild the same train/test split with the same random_state to compare
from sklearn.model_selection import train_test_split

X_train_check, X_test_check, y_train_check, y_test_check = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

print("Matches existing train encoding:", X_train_check.reset_index(drop=True).equals(
    existing_train.drop("Converted", axis=1).reset_index(drop=True)
))

Matches existing train encoding: True


In [3]:
os.makedirs("models", exist_ok=True)

# Save the dictionary of per-column encoders
joblib.dump(encoders, "models/label_encoders.pkl")

# Save the exact column order the model expects (order matters for prediction!)
feature_columns = X_encoded.columns.tolist()
with open("models/feature_columns.json", "w") as f:
    json.dump(feature_columns, f, indent=2)

# Save the mapping of category -> number for each encoder, human-readable
encoder_mappings = {
    col: {str(cls): int(idx) for idx, cls in enumerate(enc.classes_)}
    for col, enc in encoders.items()
}
with open("models/encoder_mappings.json", "w") as f:
    json.dump(encoder_mappings, f, indent=2)

print("Feature columns:", feature_columns)
print("\nEncoder mappings:")
print(json.dumps(encoder_mappings, indent=2))

Feature columns: ['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'Month', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'VisitorType', 'Weekend']

Encoder mappings:
{
  "Month": {
    "Aug": 0,
    "Dec": 1,
    "Feb": 2,
    "Jul": 3,
    "June": 4,
    "Mar": 5,
    "May": 6,
    "Nov": 7,
    "Oct": 8,
    "Sep": 9
  },
  "VisitorType": {
    "New_Visitor": 0,
    "Other": 1,
    "Returning_Visitor": 2
  }
}


In [4]:
best_rf = joblib.load("models/best_random_forest.pkl")
loaded_encoders = joblib.load("models/label_encoders.pkl")

# Take one raw row from the original (unencoded) data to simulate a "new" API request
test_row_raw = X.iloc[[0]].copy()
print("Raw input:\n", test_row_raw)

# Apply the saved encoders exactly as the API will
for col in categorical_cols:
    test_row_raw[col] = loaded_encoders[col].transform(test_row_raw[col])

# Reorder columns to match training order exactly
test_row_raw = test_row_raw[feature_columns]

prediction = best_rf.predict(test_row_raw)[0]
probability = best_rf.predict_proba(test_row_raw)[0][1]

print("\nPredicted class:", prediction)
print("Purchase probability:", probability)

Raw input:
    Administrative  Administrative_Duration  Informational  \
0               1                  14.6517              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0              19                 283.8812   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0     0.008014   0.042106   68.579222         0.0  June                 1   

   Browser  Region  TrafficType        VisitorType  Weekend  
0        2       1            1  Returning_Visitor    False  

Predicted class: 1
Purchase probability: 0.6397808062685824
